In [1]:
import pandas as pd
import numpy as np

In [2]:
A_df = pd.read_csv("data/A.csv", header=None)
B_df = pd.read_csv("data/B.csv", header=None)
C_df = pd.read_csv("data/C.csv", header=None)

# Convert all string-looking numbers to floats
A = A_df.apply(pd.to_numeric, errors='coerce').values
B = B_df.apply(pd.to_numeric, errors='coerce').values
C = C_df.apply(pd.to_numeric, errors='coerce').values

In [3]:
A_index_df = pd.read_csv("data/index_A.csv")
B_index_df = pd.read_csv("data/index_B.csv")
C_index_df = pd.read_csv("data/index_C.csv")

# Remove Transporation

In [4]:
A_transport_df = pd.read_csv("data/Transportation_A.csv")

In [5]:
# create a dict mapping each provider name to all its indices in A_index_df
mapping = A_index_df.groupby('provider name')['index'].apply(list)

# build a single flat list of all matching indices for the foreground processes
matched_indices_transport = [
    idx
    for name in A_transport_df['provider name']
    if name in mapping
    for idx in mapping[name]
]

In [6]:
import numpy as np

# matched_indices_transport is the list of indices to remove
to_drop = np.array(sorted(set(matched_indices_transport), key=int))

# 1) Remove from A_index_df
mask_keep = ~A_index_df['index'].isin(to_drop)
A_index_df = A_index_df.loc[mask_keep].copy()

# 2) Remove corresponding rows and columns from A
A = np.delete(A, to_drop, axis=0)  # remove rows
A = np.delete(A, to_drop, axis=1)  # remove columns

# 3) Remove the same columns from B (keep rows)
B = np.delete(B, to_drop, axis=1)

# 4) Reset the index column in A_index_df
A_index_df['index'] = np.arange(len(A_index_df), dtype=int)

# Remove and aggregate Electricity

In [7]:
A_elec_df = pd.read_csv("data/Electricity_A.csv")

In [8]:
# Inputs assumed:
# A : numeric numpy array (rows x cols)
# A_index_df : DataFrame with columns ["index", "provider name", "flow name", ...]
# A_elec_df : DataFrame with column ["provider name"] listing all electricity providers
# The indices in A_index_df["index"] align with both row and column positions of A.

# 0) Build the set of electricity provider names
elec_names = set(A_elec_df['provider name'].dropna().astype(str).unique())

# 1) Find their indices in A_index_df
elec_idx = A_index_df.loc[A_index_df['provider name'].astype(str).isin(elec_names), 'index'].astype(int).unique()

# 2) Locate the mix row index (must exist)
mix_name = "Electricity Mix (Global)"
mix_rows = A_index_df.loc[A_index_df['provider name'] == mix_name, 'index'].astype(int).unique()
if len(mix_rows) == 0:
    raise ValueError("Electricity Mix (Global) not found in A_index_df['provider name'].")
mix_idx = int(mix_rows[0])

# Ensure the mix row is not purged
elec_idx_set = set(map(int, elec_idx))
elec_idx_wo_mix = sorted(elec_idx_set - {mix_idx})

# 3) Aggregate: add all electricity rows (except the mix row) into the mix row, column-wise
if len(elec_idx_wo_mix) > 0:
    # in case of NaNs
    add_block = np.nansum(A[elec_idx_wo_mix, :], axis=0)
    A[mix_idx, :] = np.nan_to_num(A[mix_idx, :]) + np.nan_to_num(add_block)

# 4) Decide what to drop
rows_to_drop = np.array(elec_idx_wo_mix, dtype=int)            # drop electricity rows except the mix row
cols_to_drop = np.array(elec_idx_wo_mix, dtype=int)            # drop electricity columns except the mix column

# (Optionally also drop the mix COLUMN; keep it if you want to retain that process as a column)
# To ALSO drop the mix column, uncomment the next line:
# cols_to_drop = np.array(sorted(elec_idx_set), dtype=int)

# 5) Remove rows/columns from A and columns from B
if rows_to_drop.size > 0:
    A = np.delete(A, rows_to_drop, axis=0)
if cols_to_drop.size > 0:
    A = np.delete(A, cols_to_drop, axis=1)
    B = np.delete(B, cols_to_drop, axis=1)

# 6) Remove the same rows from A_index_df (only rows; columns in A_index_df are metadata)
if len(elec_idx_wo_mix) > 0:
    keep_mask = ~A_index_df['index'].astype(int).isin(elec_idx_wo_mix)
    A_index_df = A_index_df.loc[keep_mask].copy()

# 7) Reset the "index" column in A_index_df to reflect 0..n-1 after deletions
A_index_df['index'] = np.arange(len(A_index_df), dtype=int)

In [9]:
# 8) Zero specific entries in A for the mix column
mix_name = "Electricity Mix (Global)"
mix_col_idx_s = A_index_df.loc[A_index_df['provider name'] == mix_name, 'index'].astype(int)

if mix_col_idx_s.empty:
    raise ValueError("Mix column not found after pruning. Did you drop the mix column?")
mix_col = int(mix_col_idx_s.iloc[0])

target_names = {"Fossil Electricity", "Clean Electricity"}
row_idxs = (
    A_index_df.loc[A_index_df['provider name'].isin(target_names), 'index']
    .astype(int)
    .to_numpy()
)

if row_idxs.size > 0:
    A[row_idxs, mix_col] = 0.0
else:
    print("Warning: no rows named 'Fossil Electricity' or 'Clean Electricity' found after pruning.")

In [10]:
A[A_index_df.loc[A_index_df['provider name'] == "Fossil Electricity", 'index'].iloc[0],
  A_index_df.loc[A_index_df['provider name'] == "Electricity Mix (Global)", 'index'].iloc[0]]

0.0

In [11]:
A[A_index_df.loc[A_index_df['provider name'] == "Clean Electricity", 'index'].iloc[0],
  A_index_df.loc[A_index_df['provider name'] == "Electricity Mix (Global)", 'index'].iloc[0]]

0.0

# Remove Carbon Capture

In [12]:
carbon_capture_decisions_df = pd.read_csv("data/carbon_capture_decisions.csv")

In [13]:
import numpy as np
import pandas as pd

# normalize for reliable matching
ekey = lambda s: str(s).strip().casefold()

# copies + keys
A_index_df = A_index_df.copy()
carbon_capture_decisions_df = carbon_capture_decisions_df.copy()

A_index_df["prov_key"] = A_index_df["provider name"].map(ekey)
carbon_capture_decisions_df["col_key"] = carbon_capture_decisions_df["Provider name"].map(ekey)
carbon_capture_decisions_df["row_key"] = carbon_capture_decisions_df["Input parameters names"].map(ekey)

# provider key -> list of integer indices (row/col) in A
idx_map = (
    A_index_df.groupby("prov_key")["index"]
    .apply(lambda s: list(map(int, s)))
    .to_dict()
)

# ensure numeric X
carbon_capture_decisions_df["Linear Economy"] = pd.to_numeric(
    carbon_capture_decisions_df["Linear Economy"], errors="coerce"
)

# apply: for each decision row, A[row, col] *= X
n_pairs = 0
skipped = 0

for _, d in carbon_capture_decisions_df.iterrows():
    X = d["Linear Economy"]
    if pd.isna(X):
        skipped += 1
        continue

    row_idxs = idx_map.get(d["row_key"], [])
    col_idxs = idx_map.get(d["col_key"], [])

    if not row_idxs or not col_idxs:
        skipped += 1
        continue

    for ri in row_idxs:
        for ci in col_idxs:
            A[int(ri), int(ci)] = np.nan_to_num(A[int(ri), int(ci)]) * float(X)
            n_pairs += 1

# Remove Microplastic treatment

In [14]:
microplastics_decisions_df = pd.read_csv("data/microplastics_decisions.csv")

In [15]:
import numpy as np
import pandas as pd

# normalizer
ekey = lambda s: str(s).strip().casefold()

# copies + normalized keys
A_index_df = A_index_df.copy()
microplastics_decisions_df = microplastics_decisions_df.copy()

A_index_df["prov_key"] = A_index_df["provider name"].map(ekey)
microplastics_decisions_df["col_key"] = microplastics_decisions_df["Provider name"].map(ekey)
microplastics_decisions_df["row_key"] = microplastics_decisions_df["Input parameters names"].map(ekey)

# provider key -> list of integer indices in A (rows/cols)
idx_map = (
    A_index_df.groupby("prov_key")["index"]
    .apply(lambda s: list(map(int, s)))
    .to_dict()
)

# ensure numeric decision value
microplastics_decisions_df["No WWT Filtration"] = pd.to_numeric(
    microplastics_decisions_df["No WWT Filtration"], errors="coerce"
)

# apply decisions: A[row, col] *= value
n_pairs = 0
skipped = 0

for _, d in microplastics_decisions_df.iterrows():
    x = d["No WWT Filtration"]
    if pd.isna(x):
        skipped += 1
        continue

    row_idxs = idx_map.get(d["row_key"], [])
    col_idxs = idx_map.get(d["col_key"], [])

    if not row_idxs or not col_idxs:
        skipped += 1
        continue

    for ri in row_idxs:
        for ci in col_idxs:
            A[int(ri), int(ci)] = np.nan_to_num(A[int(ri), int(ci)]) * float(x)
            n_pairs += 1

# Dynamic LCA

In [16]:
f = np.zeros(len(A))

In [24]:
import numpy as np
import pandas as pd

# -----------------------------
# Load data
# -----------------------------
electricity_mix_df = pd.read_csv("data/electricity_mix_2025_2100.csv")
iam_indices_df     = pd.read_csv("data/iam-indices.csv")
packaging_df       = pd.read_csv("data/packaging_production_manual_2025_2100_kg.csv")

# Oil vs Natural Gas
ratio_oil_ng_df = pd.read_csv("data/ratio_oil_vs_natural_gas_2025_2100.csv")
map_oil_ng_df   = pd.read_csv("data/oil-vs-NG-feedstock.csv")

# Biomass vs Fossil
ratio_bio_fossil_df = pd.read_csv("data/ratio_biomass_vs_total_2025_2100.csv")
map_bio_fossil_df   = pd.read_csv("data/bio-vs-fossil feedstock.csv")

# -----------------------------
# Helpers
# -----------------------------
def first_idx(df, col, val):
    hit = df.loc[df[col] == val, "index"]
    return int(hit.iloc[0]) if not hit.empty else None

# Electricity Mix (Global) column index in A
mix_idx = int(A_index_df.loc[A_index_df['provider name'] == "Electricity Mix (Global)", 'index'].iloc[0])

# Map IAM electricity sources -> A row indices
iam_to_A = {}
for _, r in iam_indices_df.iterrows():
    lci_name = r["LCI Index"]
    hit = A_index_df.loc[A_index_df["provider name"] == lci_name, "index"]
    if not hit.empty:
        iam_to_A[str(r["IAM Index"])] = int(hit.iloc[0])

# -----------------------------
# Feedstock targets
# -----------------------------
feedstock_targets_oil_ng = []
for _, r in map_oil_ng_df.iterrows():
    dyn_name, prov_name, input_name = str(r["dynamic csv name"]), str(r["Provider name"]), str(r["Input parameters names"])
    row_idx = first_idx(A_index_df, "flow name", input_name) or first_idx(A_index_df, "provider name", input_name)
    col_idx = first_idx(A_index_df, "provider name", prov_name)
    if (row_idx is not None) and (col_idx is not None):
        feedstock_targets_oil_ng.append((dyn_name, row_idx, col_idx))

feedstock_targets_bio_fossil = []
for _, r in map_bio_fossil_df.iterrows():
    dyn_name, prov_name, input_name = str(r["dynamic csv name"]), str(r["Provider name"]), str(r["Input parameters names"])
    row_idx = first_idx(A_index_df, "flow name", input_name) or first_idx(A_index_df, "provider name", input_name)
    col_idx = first_idx(A_index_df, "provider name", prov_name)
    if (row_idx is not None) and (col_idx is not None):
        feedstock_targets_bio_fossil.append((dyn_name, row_idx, col_idx))

# Pre-index ratio tables
ratio_oil_ng_by_year     = ratio_oil_ng_df.set_index("Year")
ratio_bio_fossil_by_year = ratio_bio_fossil_df.set_index("Year")

# -----------------------------
# Merge electricity mix and packaging data
# -----------------------------
merged_df = electricity_mix_df.merge(packaging_df, on="Year", how="inner")

results   = []
A_by_year = {}

# -----------------------------
# Dynamic loop
# -----------------------------
for _, row in merged_df.iterrows():
    year = int(row["Year"])

    # 1) Build f
    n = A.shape[0]
    f = np.zeros(n)
    f[0] = float(row["Packaging_Plastic_kg"])

    # 2) Copy A and apply updates
    A_dyn = A.copy()

    # (a) Electricity mix: negative since it's an input
    for iam_source, idx in iam_to_A.items():
        if iam_source in row.index:
            A_dyn[idx, mix_idx] = -abs(float(row[iam_source]))/100

    # (b) Oil vs Natural Gas: negative since input
    if year in ratio_oil_ng_by_year.index:
        r_year = ratio_oil_ng_by_year.loc[year]
        for dyn_name, r_idx, c_idx in feedstock_targets_oil_ng:
            if dyn_name in r_year.index:
                A_dyn[r_idx, c_idx] = -abs(float(r_year[dyn_name]))

    # (c) Biomass vs Fossil: negative since input
    if year in ratio_bio_fossil_by_year.index:
        r_year_bf = ratio_bio_fossil_by_year.loc[year]
        for dyn_name, r_idx, c_idx in feedstock_targets_bio_fossil:
            if dyn_name in r_year_bf.index:
                A_dyn[r_idx, c_idx] = -abs(float(r_year_bf[dyn_name]))

    # 3) Solve As = f
    s = np.linalg.solve(A_dyn, f)

    # 4) g = Bs
    g = B @ s

    # 5) Store results
    results.append({
        "year": year,
        "packaging_demand_kg": f[0],
        "g_vector": g,
        "s_vector": s
    })

    # 6) Store dynamic A
    A_by_year[year] = A_dyn.copy()

In [25]:
g_2035 = next(r['g_vector'] for r in results if r['year'] == 2035)

In [26]:
s_2035 = next(r['s_vector'] for r in results if r['year'] == 2035)

In [27]:
A_2035 = A_by_year[2035]

In [28]:
A_2035[A_index_df.loc[A_index_df['provider name'] == "Electricity, nuclear (life cycle)", 'index'].iloc[0],
  A_index_df.loc[A_index_df['provider name'] == "Electricity Mix (Global)", 'index'].iloc[0]]

-0.136771

In [33]:
A_2035[A_index_df.loc[A_index_df['provider name'] == "Natural gas, processed, for material use, at plant", 'index'].iloc[0],
  A_index_df.loc[A_index_df['provider name'] == "Hydrocarbon feedstock production for use in steam cracker", 'index'].iloc[0]]

-0.5